# IDX-Trade — Decision V2 Monte Carlo
Historical development economic proxy. Not executable historical P&L.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"D:\Documents\Project\idx-v4-x1-decision-economic-comparison-20260822-v1")
EXPECTED_MANIFEST_SHA = "d33ec5ab0b6c4c7642c5faf42a7f5980f3d5e4d3d7668552d309aea0ed6e2622"
INITIAL_NAV = 50_000_000
N_PATHS = 10_000
MC_SESSIONS = 252
BLOCK = 5
SEED = 42

manifest = ROOT / "MANIFEST.json"
assert manifest.is_file(), f"Missing: {manifest}"
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA
outcomes = pd.read_csv(ROOT / "policy_signal_outcomes.csv")
summary = json.loads((ROOT / "summary.json").read_text(encoding="utf-8"))
print(summary["status"])
print(f"rows={len(outcomes):,}")


In [ ]:
def decision_v2_common_series(horizon):
    support = f"h{horizon}_complete_support"
    value = f"h{horizon}_net_proxy_primary"
    pivot = outcomes.pivot(index="date", columns="policy", values=support)
    common_dates = pivot.fillna(False).astype(bool).all(axis=1)
    dates = set(pivot.index[common_dates])
    x = outcomes.loc[
        outcomes["policy"].eq("DECISION_V2") & outcomes["date"].isin(dates),
        ["date", value],
    ].copy().sort_values("date")
    x[value] = pd.to_numeric(x[value], errors="raise")
    expected = summary["horizons"][f"H{horizon}"]["common_support_policy_metrics"]["DECISION_V2"]["net_proxy"]["PRIMARY"]
    assert len(x) == expected["n"]
    assert np.isclose(x[value].mean(), expected["mean"])
    # H5/H10 overlap. Scale each horizon outcome to a session-equivalent rate first.
    x["session_equiv"] = np.expm1(np.log1p(x[value]) / horizon)
    return x

h5 = decision_v2_common_series(5)
h10 = decision_v2_common_series(10)
pd.DataFrame({
    "H5": [len(h5), h5.iloc[:,1].mean(), h5["session_equiv"].mean()],
    "H10": [len(h10), h10.iloc[:,1].mean(), h10["session_equiv"].mean()],
}, index=["observations", "primary_net_proxy_mean", "session_equiv_mean"])


In [ ]:
def block_bootstrap(x, n_paths=N_PATHS, sessions=MC_SESSIONS, block=BLOCK, seed=SEED):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    out = np.empty((n_paths, sessions))
    for j in range(0, sessions, block):
        width = min(block, sessions - j)
        starts = rng.integers(0, len(x) - width + 1, size=n_paths)
        out[:, j:j+width] = x[starts[:, None] + np.arange(width)]
    return out

def run_mc(frame):
    r = block_bootstrap(frame["session_equiv"])
    nav = INITIAL_NAV * np.cumprod(1 + r, axis=1)
    terminal = nav[:, -1]
    peak = np.maximum.accumulate(nav, axis=1)
    mdd = (nav / peak - 1).min(axis=1)
    return nav, terminal, mdd

results = {}
for label, frame in {"H5": h5, "H10": h10}.items():
    nav, terminal, mdd = run_mc(frame)
    results[label] = (nav, terminal, mdd)

mc_summary = pd.DataFrame({
    label: {
        "terminal_nav_p05": np.quantile(t, .05),
        "terminal_nav_median": np.median(t),
        "terminal_nav_p95": np.quantile(t, .95),
        "median_252s_return": np.median(t / INITIAL_NAV - 1),
        "p_finish_below_start": np.mean(t < INITIAL_NAV),
        "median_proxy_max_drawdown": np.median(mdd),
        "p_proxy_drawdown_20pct": np.mean(mdd <= -.20),
        "p_proxy_drawdown_30pct": np.mean(mdd <= -.30),
    }
    for label, (_, t, mdd) in results.items()
}).T
mc_summary


In [ ]:
for label, (nav, terminal, _) in results.items():
    q = np.quantile(nav, [.05, .50, .95], axis=0)
    x = np.arange(1, MC_SESSIONS + 1)
    plt.figure(figsize=(10, 4))
    plt.fill_between(x, q[0], q[2], alpha=.18)
    plt.plot(x, q[1], label="median")
    plt.axhline(INITIAL_NAV, linewidth=1)
    plt.title(f"Decision V2 Monte Carlo proxy — {label}")
    plt.xlabel("session")
    plt.ylabel("notional NAV (IDR)")
    plt.legend()
    plt.show()


**Interpretation:** input = frozen Decision V2 memberships + canonical Open(t+1)→Close(t+H) target returns + PRIMARY Execution V1 friction proxy. Lot rounding, liquidity fills, historical CA quantity/cash continuity, and exact executable NAV are not reconstructed.
